# CORNEAL-TRUST: Full Cloud Training (Google Colab T4)

Trains the Phase 2 segmentation U-Net, the Phase 3 severity classifier,
and the Phase 4 image-quality + evidential severity models at full
resolution (384x384) on a free T4 GPU.

### Before starting (one-time, ~5 min)
1. In Google Drive, create a folder named `CornealTrust`.
2. Upload the 4 dataset folders into it, keeping exact names:
   `CORN-1`, `CORN-2`, `CORN-3`, `CORN1500`.
3. This notebook clones the repo from GitHub (must be public).

Then run every cell top-to-bottom (Shift+Enter). Cells 1-2 are
mandatory; the quality (Phase 4) and evidential (Phase 4) cells are
optional extras that produce 'quality_corn2.pt' and
'severity_corn1500_evidential.pt'. The setup cell force-refreshes the
repo code to origin/main, so stale checkouts cannot happen.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Setup: clone/refresh repo + copy datasets to local disk

We **copy** the datasets from Drive into the Colab VM once (about one
minute) instead of symlinking, so training reads from the fast local
disk instead of the slow Drive mount. If the repo already exists we
force it back to origin/main so it always matches the latest GitHub code.

In [ ]:
import os, sys, shutil, subprocess

# --- EDIT THIS ---
DATA_PARENT = '/content/drive/MyDrive/CornealTrust'  # folder holding CORN-1/.../CORN1500
REPO = '/content/CORNEAL_TRUST'
# -----------------

if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/gsahoo211004/CORNEAL-TRUST.git', '/content/CORNEAL_TRUST'], check=True)
else:
    print('repo exists -> force refresh to origin/main')
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/main'], check=True)
sys.path.insert(0, REPO)
for name in ['CORN-1', 'CORN-2', 'CORN-3', 'CORN1500']:
    src = os.path.join(DATA_PARENT, name)
    dst = os.path.join('/content', name)
    if os.path.exists(dst):
        print('already present:', name)
    elif os.path.exists(src):
        print('copying', name, '...')
        shutil.copytree(src, dst)
    else:
        print('!!! NOT FOUND in Drive:', src)

In [ ]:
%pip install -q -r "$REPO/requirements.txt" 2>&1 | tail -3
# imagecodecs decodes the LZW/PackBits TIFFs (safety net if above missed it)
%pip install -q imagecodecs 2>&1 | tail -2
print('deps installed')

In [ ]:
%cd "$REPO"
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('cpu count:', os.cpu_count())

## 1. Segmentation U-Net (Phase 2) -- CORN-1
50-epoch run, early stopping on validation Dice. Optional if you already
have a good unet_corn1.pt -- skip this cell by just not running it.
Takes ~20-40 min. `--num-workers 2` matches the Colab 2-vCPU runtime.

In [ ]:
!python scripts/train_segmentation.py --num-workers 2

import glob
print(glob.glob('outputs/checkpoints/unet_corn1.pt'))

## 2. Severity classifier (Phase 3) -- CORN1500 + CORN-3 val
60-epoch run, patience 20, lr 2e-4, class-balanced CE. Takes ~30-40 min.

In [ ]:
!python scripts/train_severity.py --num-workers 2 --epochs 60 --patience 20 --lr 0.0002 --class-balanced

import glob
print(glob.glob('outputs/checkpoints/severity_corn1500.pt'))

## 3. Quality classifier (Phase 4, optional) -- CORN-2
Binary high/low quality model feeding the CDTI Q_image term. Takes ~5-10 min.

In [ ]:
!python scripts/train_quality.py --num-workers 2

import glob
print(glob.glob('outputs/checkpoints/quality_corn2.pt'))

## 4. Evidential severity (Phase 4, optional) -- CORN1500 + CORN-3 val
Dirichlet (uncertainty-aware) head. Same speed as Phase 3 severity.
Produces severity_corn1500_evidential.pt used with --evidential + CDTI inference.

In [ ]:
!python scripts/train_severity.py --evidential --num-workers 2

import glob
print(glob.glob('outputs/checkpoints/severity_corn1500_evidential.pt'))

In [ ]:
# Safety copy: mirror checkpoints + logs back to Drive
import os, shutil, glob
out = os.path.join(DATA_PARENT, 'CORNEAL_TRUST_outputs')
shutil.copytree('outputs', os.path.join(out, 'outputs'), dirs_exist_ok=True)
print('saved to', out)
for ckpt in ['unet_corn1.pt', 'severity_corn1500.pt', 'quality_corn2.pt', 'severity_corn1500_evidential.pt']:
    hit = glob.glob(f'outputs/checkpoints/{ckpt}')
    print(('OK  ' if hit else 'MISS'), ckpt)
print('\nIMPORTANT: download the present checkpoints from MyDrive/CornealTrust/CORNEAL_TRUST_outputs/outputs/checkpoints/.')